# Audit 01 — VD inventory（Colab）

2026-09 資料重新盤點的第一步：把 Drive 上**整段** `vd_station_5m`（2021-06 ~ 2026-08，約 1,759 天）掃一遍，回答三個問題：

1. 哪些天有檔、哪些天缺檔（station 與 raw 兩層）。
2. 每一天每一個 VD 回報了多少 5 分鐘 slot（validity）、平均速度、有效車道數、流量。
3. 走廊（N1 100.8–251.1 km、N3 110.9–269.0 km）逐月、逐公里的覆蓋率，以及「鎖版 2024-04~2025-05」vs「延伸到 2023-08~2026-04」兩種 cohort 視窗的資料品質對照。

所有運算都在 `scripts/analysis/audit_vd_inventory.py`，notebook 只是殼。輸出寫到 `output/data_audit/vd_inventory/`（在 Drive 上、gitignored），**不會動到任何鎖版資料或 `reports/`**。

| 章節 | 做什麼 | 約略時間 |
| --- | --- | --- |
| 1 | 掛載 Drive、找專案根目錄、裝 `uv` | 1 分鐘 |
| 2 | 逐日掃描（可中斷、可續跑） | 本機 Drive streaming 約 12 s/天；Colab 掛載預期快很多，全量估 30–90 分鐘 |
| 3 | 看摘要表與圖 | 1 分鐘 |
| 4 | 深入：異常日、缺檔日、低品質 VD | 依需要 |

> 掃描是 **resumable**：每天寫一個 `daily/vd_day_YYYYMMDD.parquet`，Colab 斷線後重跑第 2 節即可接續。

## 1. 掛載 Drive 與專案根目錄

Drive 資料夾名稱在 2026-09 前後有兩種（`陽交大高速公路計畫/main` 或直接 `main`），下面的 cell 兩種都會找。若你是被分享者，先在 Drive 網頁把資料夾「新增捷徑到我的雲端硬碟」。

In [ ]:
import os, sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)


def _is_root(p):
    return (Path(p) / 'CLAUDE.md').is_file() and (Path(p) / 'scripts' / 'analysis' / 'audit_vd_inventory.py').is_file()


def _candidates():
    if os.environ.get('PROJECT_ROOT'):
        yield Path(os.environ['PROJECT_ROOT'])
    if IN_COLAB:
        my = Path('/content/drive/MyDrive')
        yield my / 'main'
        yield my / '陽交大高速公路計畫' / 'main'
        if my.is_dir():
            for child in sorted(my.iterdir()):
                if child.is_dir():
                    yield child / 'main'
                    yield child / '陽交大高速公路計畫' / 'main'
        for shared in ('Shareddrives', 'Shared drives'):
            sd = Path('/content/drive') / shared
            if sd.is_dir():
                for child in sorted(sd.iterdir()):
                    yield child / 'main'
                    yield child / '陽交大高速公路計畫' / 'main'
    else:
        here = Path(os.getcwd()).resolve()
        for p in [here, *here.parents]:
            yield p


PROJECT_ROOT = None
seen = set()
for cand in _candidates():
    if cand in seen:
        continue
    seen.add(cand)
    try:
        if _is_root(cand):
            PROJECT_ROOT = cand
            break
    except OSError:
        pass
if PROJECT_ROOT is None:
    raise SystemExit('找不到專案根目錄。若是被分享者請先新增捷徑到「我的雲端硬碟」；或手動設定 os.environ["PROJECT_ROOT"] 後重跑。')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
OUT_DIR = PROJECT_ROOT / 'output' / 'data_audit' / 'vd_inventory'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUT_DIR      =', OUT_DIR)
for p in ['data/vd_station_5m', 'data/vd_raw_lane_1m_full', 'central_vd_list.txt']:
    print(f'{(PROJECT_ROOT / p).exists()!s:>5}  {p}')

In [ ]:
# 安裝 uv；script 用 inline /// script /// 依賴區塊自帶 duckdb / pandas / pyarrow / matplotlib
!pip install -q uv
!uv --version

## 2. 逐日掃描（可續跑）

預設掃描 `vd_station_5m` 內**全部**存在的日期。第一次跑建議先用 `--limit-days 30` 看速度，再拿掉全量跑。

- `--resume`：略過已經有 `daily/vd_day_*.parquet` 的日子（斷線重跑就靠這個）。
- `--start-date/--end-date`：只掃一段。
- `--summary-only`：不掃描，只用既有 daily/ 重算摘要與圖。
- `--threads 2 --memory-limit 6GB`：Colab 免費機型的保守設定。

DuckDB 一天只讀一個 ~4 MB Parquet，記憶體壓力很小；瓶頸是 Drive I/O。

In [ ]:
# 先試 30 天，確認速度與輸出格式
!uv run scripts/analysis/audit_vd_inventory.py --resume --limit-days 30 --threads 2 --memory-limit 6GB --no-plots

In [ ]:
# 全量掃描（斷線後重新執行此 cell 即可接續）
!uv run scripts/analysis/audit_vd_inventory.py --resume --threads 2 --memory-limit 6GB

## 3. 摘要與圖

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown((OUT_DIR / 'SUMMARY.md').read_text(encoding='utf-8')))

In [ ]:
for name in sorted((OUT_DIR / 'figures').glob('*.png')):
    print(name.name)
    display(Image(filename=str(name), width=1100))

## 4. 深入看：缺檔日、異常日、低品質 VD、中央清單裡從未出現的 VD

判讀重點：

- **缺檔日** 若集中在某幾個月，代表當時下載失敗或來源缺資料；要不要補抓看 `raw_exists` 欄（raw 有、station 沒有 → 叩需重跑 station build）。
- **異常日**（走廊平均 validity < 0.5，或走廊 VD 數 < 中位數 70%）多半是 MOTC 來源整天壞掉，事故樣本落在這些天會被 `filled_ratio < 0.8` 剔除；延伸 cohort 前要知道會損失多少。
- **低品質 VD**：`mean_valid_ratio` 低或 `presence_ratio_overall` 低的走廊 VD，決定 1-km 格子的 filled_ratio。
- **中央清單從未出珝的 VD**：可能已撤站或改 ID，`central_vd_list.txt` 需要更新。

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

monthly = pd.read_csv(OUT_DIR / 'monthly_summary.csv')
missing = pd.read_csv(OUT_DIR / 'missing_days.csv')
anomaly = pd.read_csv(OUT_DIR / 'anomaly_days.csv')
vd = pd.read_csv(OUT_DIR / 'vd_summary.csv')

print('=== monthly ===')
display(monthly)
print(f'=== missing days: {len(missing)} ===')
display(missing.assign(month=missing['date'].str[:7]).groupby('month').size().rename('n_missing').to_frame().T)
print(f'=== anomaly days: {len(anomaly)} ===')
display(anomaly[['date', 'weekday', 'n_vd_corridor', 'mean_valid_ratio_corridor', 'share_corridor_vd_ge80']])

In [ ]:
corr = vd[vd['in_corridor']].copy()
print(f'corridor VDs: {len(corr)}; mean valid ratio < 0.8: {(corr["mean_valid_ratio"] < 0.8).sum()}; present < 50% of days: {(corr["presence_ratio_overall"] < 0.5).sum()}')
display(corr.sort_values('mean_valid_ratio').head(25)[['vdid', 'freeway', 'direction', 'mileage_km', 'first_date', 'last_date', 'n_days_present', 'presence_ratio_overall', 'mean_valid_ratio', 'p10_valid_ratio', 'mean_lanes_valid']])

In [ ]:
central = [ln.strip() for ln in (PROJECT_ROOT / 'central_vd_list.txt').read_text(encoding='utf-8').splitlines() if ln.strip()]
never_seen = sorted(set(central) - set(vd['vdid']))
print(f'central list: {len(central)}; never seen in station data: {len(never_seen)}')
print('\n'.join(never_seen[:40]))

In [ ]:
# 走廊 1-km × 月：哪些 (freeway, direction, km) 在哪些月份沒有任何 VD ≥ 0.8
km = pd.read_csv(OUT_DIR / 'corridor_km_month.csv')
weak = km[km['best_vd_valid_ratio'] < 0.8]
print(f'(freeway, direction, km, month) cells with no VD reaching 0.8: {len(weak)} of {len(km)}')
display(weak.groupby(['freeway', 'direction']).size().rename('n_weak_cells'))
display(weak.groupby('year_month').size().rename('n_weak_cells').to_frame().T)

## 5. 下一步

把 `SUMMARY.md` 的 cohort 視窗對照表與這裡的判讀寫進 `docs/DATA_AUDIT_<日期>.md`，然後決定：

1. 延伸 cohort 到 2023-08 ~ 2026-04 是否可行（缺檔日 / 異常日比例可接受？）。
2. `central_vd_list.txt` 是否要更新（撤站、改 ID）。
3. 是否需要補下載 raw 或重跑 station build 補缺檔日。

接著才是事故來源對照（dispatch / SMS / LINE）、天氣與 ETC 的評估。